# P2.1 — Semantic LoRA — Google Colab + Google Drive

Notebook này lưu checkpoint, file kết quả và cache mô hình trong `MyDrive/SeqLogAD/P2.1_Semantic`.

## Chuẩn bị

1. Trong Colab, chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU có VRAM cao hơn). GPU Colab thay đổi theo thời điểm; bước kiểm tra bên dưới sẽ xác nhận CUDA và khả năng chạy. BF16 được ưu tiên, FP16 chỉ dùng theo chính sách đã có trong mã nguồn. Nếu preflight thiếu VRAM hoặc phát hiện lỗi số học, notebook sẽ dừng.
2. Chạy cell đầu để kết nối Google Drive. Đưa hai ZIP vào `MyDrive/SeqLogAD/P2.1_Semantic/uploads/`:
   - `seqlogad-code-s42-hdfs-bgl-v3.zip`
   - `phase2-final.zip`
3. Thêm Hugging Face token có quyền truy cập `meta-llama/Llama-3.1-8B` vào **Colab Secrets** với tên `HF_TOKEN`, rồi bật quyền truy cập secret cho notebook. Không dán token vào cell.
4. Chọn **Runtime → Run all**.

Mã nguồn và môi trường tạm được giải nén/cài ở `/content` để tránh I/O chậm trên Drive. Checkpoint phục hồi, cache mô hình và kết quả cuối được lưu trong `MyDrive/SeqLogAD/P2.1_Semantic`. Nếu Colab ngắt, kết nối lại Drive rồi chạy lại notebook để tiếp tục từ checkpoint hợp lệ. Chỉ ngắt phiên sau khi thấy `SAFE_TO_DISCONNECT_COLAB=YES`.

Đây là lượt chạy **development seed 42**, huấn luyện mới cả hai fold target **BGL** và **HDFS** (EXC-007: Hadoop và OpenStack chỉ làm source). Kết quả chưa phải đánh giá khoa học cuối cùng.


In [ ]:
# HUMAN CONFIG — seed/fold are fixed by the development protocol.
from pathlib import Path
import os, uuid
from google.colab import drive

drive.mount("/content/drive")

TASK_ID = "P2.1"
NOTEBOOK_STATUS = "READY"
BLOCKERS = []
RUN_MODE = "development"
SEED = 42
TARGETS = ["BGL", "HDFS"]  # EXC-007: Hadoop/OpenStack are source-only
DRIVE_ROOT = Path("/content/drive/MyDrive/SeqLogAD/P2.1_Semantic")  # P2.2 uses its own root: .../P2.2_Sequence
UPLOAD_ROOT = DRIVE_ROOT / "uploads"
PERSISTENT_ROOT = DRIVE_ROOT
HF_HOME = DRIVE_ROOT / "hf-cache"
# Keep Python, extracted source and temporary outputs on Colab's local disk.
STATE_ROOT = Path("/content/SeqLogAD")
VENV = STATE_ROOT / "venv-py312"
RUNTIME_ROOT = STATE_ROOT / "runtime" / "feef719adcde-ebcfdfd6eb7b"
REPO_ROOT = str(RUNTIME_ROOT / "seqlogad_code_all_experts_s42")
DATA_ROOT = str(RUNTIME_ROOT / "phase2")
SESSION_ROOT = Path("/content/seqlogad_sessions") / uuid.uuid4().hex
OUTPUT_ROOT = str(SESSION_ROOT / "seqlogad_outputs")
EXPECTED_CODE_ZIP_SHA256 = "feef719adcde698b507a9b0ffe46e97ca178371cc8c3d215dfd16f26b758edba"
EXPECTED_DATA_ZIP_SHA256 = "ebcfdfd6eb7b1607eb6d57e60fae88456d03bcb8ae2c6688ea398042c856224b"
EXPECTED_BUNDLE_SHA256 = "3ef2bfe36d2ad68205eea4290a632eb81db0d63acfb76c92f25a3eb96e8205de"

assert RUN_MODE == "development" and SEED == 42
assert TARGETS == ["BGL", "HDFS"]
assert NOTEBOOK_STATUS == "READY", "Notebook is fail-closed: " + ", ".join(BLOCKERS)
UPLOAD_ROOT.mkdir(parents=True, exist_ok=True)
for directory in (STATE_ROOT, PERSISTENT_ROOT, HF_HOME, SESSION_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
print({"session_root": str(SESSION_ROOT), "persistent_root": str(PERSISTENT_ROOT),
       "upload_folder": str(UPLOAD_ROOT)})


In [ ]:
# Colab normally includes uv; install it only if absent.
import sys, shutil, subprocess

uv_real = shutil.which("uv")
if not uv_real:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.8.17"], check=True)
    uv_real = shutil.which("uv")
assert uv_real, "uv installation failed"
uv_expected = Path(sys.executable).parent / "uv"
if not uv_expected.exists():
    uv_expected.symlink_to(uv_real)
print("uv:", uv_real)


## Verify uploads, extract atomically and build the frozen runtime


In [ ]:
from pathlib import PurePosixPath
import hashlib, json, shutil, stat, subprocess, sys, zipfile

def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

def locate(expected):
    matches = [path for path in UPLOAD_ROOT.glob("*.zip") if digest(path) == expected]
    assert len(matches) == 1, (
        f"Put the matching ZIP in MyDrive/SeqLogAD/P2.1_Semantic/uploads; expected SHA-256 {expected}; "
        f"found {matches}"
    )
    return matches[0]

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as handle:
        infos = handle.infolist()
        names = [item.filename for item in infos]
        assert len(names) == len(set(names)), "Duplicate ZIP member"
        for item in infos:
            member = PurePosixPath(item.filename)
            assert not member.is_absolute() and ".." not in member.parts and "\\" not in item.filename
            assert ((item.external_attr >> 16) & 0o170000) != stat.S_IFLNK, "ZIP symlink forbidden"
        assert handle.testzip() is None, "ZIP CRC failure"
        handle.extractall(destination)

code_zip = locate(EXPECTED_CODE_ZIP_SHA256)
data_zip = locate(EXPECTED_DATA_ZIP_SHA256)
marker = RUNTIME_ROOT / "EXTRACTION.json"
expected_marker = {
    "code_sha256": EXPECTED_CODE_ZIP_SHA256,
    "data_sha256": EXPECTED_DATA_ZIP_SHA256,
}
if RUNTIME_ROOT.exists():
    assert marker.is_file(), f"Unverified runtime directory exists: {RUNTIME_ROOT}"
    assert json.loads(marker.read_text()) == expected_marker, "Runtime extraction identity mismatch"
else:
    partial = RUNTIME_ROOT.with_name(RUNTIME_ROOT.name + ".partial")
    if partial.exists():
        shutil.rmtree(partial)
    partial.mkdir(parents=True)
    safe_extract(code_zip, partial)
    safe_extract(data_zip, partial)
    assert (partial / "seqlogad_code_all_experts_s42").is_dir()
    assert (partial / "phase2/manifests/bundle.json").is_file()
    (partial / "EXTRACTION.json").write_text(json.dumps(expected_marker, sort_keys=True) + "\n")
    partial.rename(RUNTIME_ROOT)

assert Path(REPO_ROOT).is_dir()
assert Path(DATA_ROOT, "manifests/bundle.json").is_file()
print({"code_zip": str(code_zip), "data_zip": str(data_zip), "extraction": "PASS"})

def isolated_env():
    env = dict(os.environ)
    env.pop("PYTHONPATH", None)
    env["PYTHONNOUSERSITE"] = "1"
    env["PYTHONUNBUFFERED"] = "1"
    return env

def run_live(args, check=True):
    with subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=isolated_env()) as process:
        for line in process.stdout:
            token = os.environ.get("HF_TOKEN")
            if token:
                line = line.replace(token, "[TOKEN_HIDDEN]")
            print(line, end="", flush=True)
        code = process.wait()
    if check and code:
        raise subprocess.CalledProcessError(code, args)
    return code

run_live(["nvidia-smi"])
PYTHON = str(VENV / "bin/python")
if not Path(PYTHON).is_file():
    if sys.version_info[:2] == (3, 12):
        run_live([sys.executable, "-m", "pip", "install", "virtualenv==20.31.2"])
        run_live([sys.executable, "-m", "virtualenv", "--python", sys.executable, str(VENV)])
    else:
        run_live([sys.executable, "-m", "pip", "install", "uv==0.8.17"])
        uv = str(Path(sys.executable).parent / "uv")
        assert Path(uv).is_file(), "uv bootstrap executable missing"
        run_live([uv, "python", "install", "3.12"])
        run_live([uv, "venv", "--python", "3.12", "--seed", str(VENV)])

run_live([PYTHON, "-c", "import sys; assert sys.version_info[:2] == (3, 12); print(sys.version)"])
run_live([PYTHON, "-m", "pip", "install", "torch==2.9.1", "--index-url",
          "https://download.pytorch.org/whl/cu128"])
run_live([PYTHON, "-m", "pip", "install", "-e", REPO_ROOT, "packaging>=24,<27"])
run_live([PYTHON, "-c", "from seqlogad.semantic.runtime import install_missing; install_missing(" + repr(REPO_ROOT) + ")"])
hardware_check = f"""
import json, torch
assert torch.cuda.is_available(), "CUDA is unavailable"
name = torch.cuda.get_device_name(0)
free, total = torch.cuda.mem_get_info(0)
native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
capability = torch.cuda.get_device_capability(0)
assert capability >= (7, 5), f"GPU compute capability {{capability}} is below the supported minimum (7.5)"
print(json.dumps({{"gpu": name, "compute_capability": capability, "free_vram_bytes": free, "total_vram_bytes": total,
                  "native_bf16": native_bf16, "precision_policy": "BF16 if natively supported; otherwise conditional FP16", "torch": torch.__version__,
                  "cuda": torch.version.cuda}}, indent=2))
"""
run_live([PYTHON, "-c", hardware_check])
run_live([PYTHON, "-m", "seqlogad.semantic.cli", "environment", "--repo-root", REPO_ROOT])


## Secure Hugging Face authentication


In [ ]:
# Read the token from Colab Secrets; it is never printed or written to Drive.
from google.colab import userdata

token = os.environ.get("HF_TOKEN")
if not token:
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
assert token, (
    "Add HF_TOKEN in Colab Secrets, enable notebook access, then rerun this cell"
)
run_live([PYTHON, "-m", "seqlogad.semantic.cli", "auth", "--repo-root", REPO_ROOT])


## Sequential target loop with persistent recovery


In [ ]:
args = [
    PYTHON, "-m", "seqlogad.phase2_execution.cli",
    "--task-id", TASK_ID,
    "--repo-root", REPO_ROOT,
    "--data-root", DATA_ROOT,
    "--output-root", OUTPUT_ROOT,
    "--drive-root", str(PERSISTENT_ROOT),
    "--expected-bundle-sha256", EXPECTED_BUNDLE_SHA256,
]
run_live(args, check=True)


## Final persistent-integrity and safe-stop gate


In [ ]:
final_zip = PERSISTENT_ROOT / TASK_ID / "seqlogad_outputs_P2.1_S42_FINAL.zip"
final_sidecar = Path(str(final_zip) + ".sha256")
assert final_zip.is_file() and final_sidecar.is_file(), "Verified final persistent ZIP is missing"
declared_hash, declared_name = final_sidecar.read_text().strip().split("  ", 1)
assert declared_name == final_zip.name and digest(final_zip) == declared_hash
# Semantic output must hold one freshly trained adapter archive per target, no external registration.
with zipfile.ZipFile(final_zip) as handle:
    members = set(handle.namelist())
expected_targets = {f"P2.1_S42/targets/P2.1_{t}_S42.zip" for t in TARGETS}
assert expected_targets <= members, f"Missing semantic target archives: {sorted(expected_targets - members)}"
assert not any(m.endswith("REGISTERED_EXTERNALLY.json") for m in members), "Unexpected external registration"
shutil.rmtree(SESSION_ROOT)
print("====================================")
print("SEQLOGAD COLAB EXECUTION SUMMARY — P2.1 SEMANTIC")
print(f"SEMANTIC_TARGETS={TARGETS}")
print(f"FINAL_ZIP={final_zip}")
print(f"FINAL_ZIP_SIZE={final_zip.stat().st_size}")
print(f"FINAL_ZIP_SHA256={declared_hash}")
print("PERSISTENT_STORAGE_VERIFIED=YES")
print("SAFE_TO_DISCONNECT_COLAB=YES")
print("====================================")
